In [ ]:
from pathlib import Path
path_spec_data=Path.cwd().parent.parent/"spec_data"
path_benchmark_data=Path.cwd().parent.parent/"benchmark_search_result"

path_spec_data.mkdir(parents=True, exist_ok=True)
path_benchmark_data.mkdir(parents=True, exist_ok=True)

In [2]:
# Only record hybrid search

original_library_size = [
    1_000_000, ]
    
query_size = 100

update_size=1_000

num_per_group=10_000_000
cache_list_threshold=1_000_000
ion_mode = [-1, 1]

steps=["build", "open_search", "neutral_loss_search", "hybrid_search"]

construction_type=['dynamic_all_build', 'dynamic_build_update', 'flash_all_build']

script='24_compare_search_result.py'


In [ ]:
import subprocess
import pickle
import os
import time
import shutil
from typing import Union
import numpy as np
import msgpack
from dynamic_entropy_search.dynamic_entropy_search import DynamicEntropySearch
from dynamic_entropy_search.convert_to_mgf import _write_spec


def run_usrbintime_by_arguments(
          arguments:list[str], 
          query_size:int,
          charge:int,
          if_output:bool=False, 
          

          ):
    
    # arguments: script_path, str(charge), step
    # command=["python"] + arguments
    command=["python"] + arguments
    if if_output: # Output to files as record
        output_file_name=path_benchmark_data/f'compare_search_result_query_{query_size}_charge_{charge}.txt'
        with open(output_file_name, 'w') as f1:
            subprocess.run(command, stderr=f1, stdout=f1, cwd=Path.cwd(), env=os.environ.copy(), text=True)
        return
    else: # Output is not needed
         
        subprocess.run(command, cwd=Path.cwd(), env=os.environ.copy())
        return
    



for library_size in original_library_size:
    for charge in ion_mode:
        query_pkl=path_spec_data/f"benchmark_spec/query_spectra-charge_{charge}-number_100.pkl"

        for step in steps:
            if step=='build':

                ### DynamicEntropySearch all_build ###
                # Remove the old index
                path_comparison_dynamic_data=Path.cwd().parent.parent/f"comparison_data/dynamic_all_build/charge-{charge}"
                if path_comparison_dynamic_data.exists():
                    shutil.rmtree(path_comparison_dynamic_data)

                ### DynamicEntropySearch build_0.1_update_0.9*1 ###
                # Remove the old index
                path_comparison_dynamic_data=Path.cwd().parent.parent/f"comparison_data/dynamic_build_update_1/charge-{charge}"
                if path_comparison_dynamic_data.exists():
                    shutil.rmtree(path_comparison_dynamic_data)

                ### DynamicEntropySearch build_0.1_update_0.09*10 ###
                # Remove the old index
                path_comparison_dynamic_data=Path.cwd().parent.parent/f"comparison_data/dynamic_build_update_2/charge-{charge}"
                if path_comparison_dynamic_data.exists():
                    shutil.rmtree(path_comparison_dynamic_data)

                ### FlashEntropySearch all_build ###
                # Remove the old index
                path_comparison_flash_data=Path.cwd().parent.parent/f"comparison_data/flash_all_build/charge-{charge}"
                if path_comparison_flash_data.exists():
                    shutil.rmtree(path_comparison_flash_data)

                arguments=[script, str(charge), query_pkl]
                
                run_usrbintime_by_arguments(arguments=arguments, charge=charge, query_size=query_size, if_output=True)




                
                            
